# 02. Quantized Attention: SageAttention INT8 and FP8

- **Category:** 14_quantisation / 05_kv_attention_optim_and_benchmarks
- **Lineage & Citations:** Zhang et al. (2024, *SageAttention: Accurate and Efficient 8-Bit Attention without Retraining*).
- **Purpose:** Enable accurate 8-bit attention computation ($Q K^\top$ in INT8 and $P V$ in INT8/FP8) by proving and exploiting the exact row-wise invariance of softmax under Key-mean centering.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [01. KV Cache Quantization KIVI and KVQuant](01_kv_cache_quantization_kivi_and_kvquant.ipynb) $\to$ `🔗 Downstream Syntheses:` [03. Quantized Optimizer States 8bit and 4bit Adam](03_quantized_optimizer_states_8bit_and_4bit_adam.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT subtract Query mean $\bar{Q}$ instead of Key mean $\bar{K}$ (Query centering adds column-varying offsets that do NOT cancel in row-wise softmax!); do NOT apply uncentered INT8 quantization to Keys (large DC channel bias reduces effective INT8 resolution to $<3$ bits).

## Mathematical Foundations & Softmax Invariance Theorem

1. **Key-Mean Centering**:
   $$\widetilde{\mathbf{K}} = \mathbf{K} - \bar{\mathbf{K}}, \quad \bar{\mathbf{K}} = \frac{1}{N} \sum_{t=1}^N \mathbf{K}_t$$
2. **Softmax Invariance Theorem**:
   $$\mathbf{q}_i \widetilde{\mathbf{K}}_j^\top = \mathbf{q}_i \mathbf{k}_j^\top - \mathbf{q}_i \bar{\mathbf{k}}^\top = S_{ij} - C_i$$
   $$\text{softmax}(\mathbf{S}_i - C_i)_j = \frac{e^{S_{ij} - C_i}}{\sum_k e^{S_{ik} - C_i}} = \frac{e^{-C_i} e^{S_{ij}}}{e^{-C_i} \sum_k e^{S_{ik}}} \equiv \text{softmax}(\mathbf{S}_i)_j$$
3. **Integer GEMM Rescaling**:
   $$\mathbf{S}_{\text{sage}} = (\mathbf{Q}_{\text{int8}} \widetilde{\mathbf{K}}_{\text{int8}}^\top) \cdot \left( \frac{s_Q \cdot s_K}{\sqrt{d_k}} \right), \quad \mathbf{O} = \text{softmax}(\mathbf{S}_{\text{sage}}) \cdot \mathbf{V}_{\text{deq}}$$

In [ ]:
import sys, time, math
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/05_kv_attention_optim_and_benchmarks/02_quantized_attention_sageattention_int8_fp8.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def key_mean_smoothing(k: torch.Tensor):
    k_mean = torch.mean(k, dim=0, keepdim=True)
    return k - k_mean, k_mean

def quantize_symmetric_int8(x: torch.Tensor, eps: float = 1e-7):
    scale = torch.clamp(torch.max(torch.abs(x)) / 127.0, min=eps)
    x_int8 = torch.clamp(torch.round(x / scale), -127.0, 127.0)
    return x_int8 * scale, x_int8, scale.reshape(1)

def sage_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):
    d_k = q.shape[1]
    k_smoothed, k_mean = key_mean_smoothing(k)
    q_deq, q_int8, s_q = quantize_symmetric_int8(q)
    k_deq, k_int8, s_k = quantize_symmetric_int8(k_smoothed)
    int_acc = q_int8 @ k_int8.T
    scores = int_acc * ((s_q * s_k) / math.sqrt(float(d_k)))
    probs = torch.softmax(scores, dim=-1)
    v_deq, v_int8, s_v = quantize_symmetric_int8(v)
    return probs @ v_deq, scores, probs, int_acc, k_smoothed

In [ ]:
# 1. Linear anchor parity
assert float(torch.max(torch.abs(weights["test_input"] @ weights["weight"] + weights["bias"] - weights["test_output"]))) < 1e-5

# 2. Key-mean smoothing parity
k_smoothed, k_mean = key_mean_smoothing(weights["k"])
assert float(torch.max(torch.abs(k_smoothed - weights["k_smoothed"]))) < 1e-5
assert float(torch.max(torch.abs(k_mean - weights["k_mean"]))) < 1e-5

# 3. INT8 quantization parity
q_deq, q_int8, s_q = quantize_symmetric_int8(weights["q"])
assert float(torch.max(torch.abs(q_deq - weights["q_deq"]))) < 1e-5
assert float(torch.max(torch.abs(q_int8 - weights["q_int8"]))) < 1e-5
assert float(torch.max(torch.abs(s_q - weights["s_q"]))) < 1e-5

# 4. SageAttention forward parity
out_sage, scores_sage, probs_sage, int_acc, _ = sage_attention(weights["q"], weights["k"], weights["v"])
assert float(torch.max(torch.abs(int_acc - weights["int_acc"]))) < 1e-5
assert float(torch.max(torch.abs(scores_sage - weights["scores_sage"]))) < 1e-5
assert float(torch.max(torch.abs(probs_sage - weights["probs_sage"]))) < 1e-5
assert float(torch.max(torch.abs(out_sage - weights["out_sage"]))) < 1e-5
print("All SageAttention Key-mean smoothing and forward assertions verified against golden checkpoint!")

In [ ]:
# Softmax Invariance Theorem & Output SQNR Validation
diff_inv = float(torch.max(torch.abs(weights["probs_fp32_smooth"] - weights["probs_fp32_raw"])))
print(f"Softmax Invariance Empirical Max Diff: {diff_inv:.2e}")
assert diff_inv < 1e-6, f"Softmax Invariance must hold to numerical precision (<1e-6), got {diff_inv:.2e}"

y_fp32 = weights["out_fp32_raw"]
mse = float(torch.mean((y_fp32 - weights["out_sage"]) ** 2))
sig_pwr = float(torch.mean(y_fp32 ** 2))
sqnr = 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()
print(f"SageAttention INT8/FP8 SQNR vs FP32: {sqnr:.2f} dB")
assert sqnr > 35.0, "SageAttention must achieve >35 dB output SQNR!"

In [ ]:
# Latency Microbenchmark: SageAttention Forward Pass (Batch=16, Seq=256, D=64)
b_bench, s_bench, d_bench = 16, 256, 64
q_bench = torch.randn(b_bench, d_bench)
k_bench = torch.randn(s_bench, d_bench) + 2.0
v_bench = torch.randn(s_bench, d_bench)

t0 = time.perf_counter()
for _ in range(50):
    _ = sage_attention(q_bench, k_bench, v_bench)
t_sage = (time.perf_counter() - t0) / 50 * 1e3

print(f"SageAttention (B={b_bench}, S={s_bench}, D={d_bench}) Latency: {t_sage:.3f} ms")
assert t_sage < 25.0, "SageAttention must complete in <25 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Exact Mathematical Invariance**: Subtracting Key sequence mean $\bar{K}$ injects a row-constant scalar offset $C_i = Q_i \bar{K}^\top$ that cancels out exactly in row-wise softmax.
2. **Elimination of DC Outliers**: Centering keys shrinks the activation dynamic range, enabling high-precision INT8 Tensor Core execution for $Q \widetilde{K}^\top$.
3. **Speedup & Efficiency**: SageAttention accelerates attention by up to $2.5\times$ in long-context inference and DiT video generation without offline calibration or fine-tuning.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_quantized_attention_sageattention_int8_fp8.md](../../../markdown/14_quantisation/05_kv_attention_optim_and_benchmarks/02_quantized_attention_sageattention_int8_fp8.md)
- **PyTorch 2.x Module:** [02_quantized_attention_sageattention_int8_fp8.py](../../../pytorch2/14_quantisation/05_kv_attention_optim_and_benchmarks/02_quantized_attention_sageattention_int8_fp8.py)
- **Functional JAX Module:** [02_quantized_attention_sageattention_int8_fp8.py](../../../jax/14_quantisation/05_kv_attention_optim_and_benchmarks/02_quantized_attention_sageattention_int8_fp8.py)
- **Native Rust Candle Track:** [02_quantized_attention_sageattention_int8_fp8.rs](../../../candle/14_quantisation/05_kv_attention_optim_and_benchmarks/02_quantized_attention_sageattention_int8_fp8.rs)
- **Hardware Kernel (CPU / WGSL):** [02_quantized_attention_sageattention_int8_fp8.rs](../../../kernels/14_quantisation/05_kv_attention_optim_and_benchmarks/02_quantized_attention_sageattention_int8_fp8.rs)
- **Golden SafeTensors:** [02_quantized_attention_sageattention_int8_fp8.safetensors](../../../weights/14_quantisation/05_kv_attention_optim_and_benchmarks/02_quantized_attention_sageattention_int8_fp8.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. KV Cache Quantization KIVI and KVQuant](01_kv_cache_quantization_kivi_and_kvquant.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 03. Quantized Optimizer States 8bit and 4bit Adam](03_quantized_optimizer_states_8bit_and_4bit_adam.ipynb) &rarr;